# Appendix: Multi-Task Neural Net (FM-MLP)

This notebook explores whether a single joint multi-task neural network (shared
FM+MLP backbone, dual churn/forward-revenue heads) can beat the project's
primary model — CatBoost churn/forward-revenue + Cox survival, in
`03_CatBoost_and_Cox_Models.ipynb`. It merges what were three separate
notebooks (Model Architecture, Training Baselines, Multi-Task Ablation) into
three parts below, ending in a direct comparison against the CatBoost
headline.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_curve

from kkbox import config as kkbox_config
from kkbox.data import columns_from_manifest, load_splits, make_loader, stratified_subsample
from kkbox.determinism import seed_everything
from kkbox.metrics import evaluate_full
from kkbox.models import FMInteractionLayer, MultiTaskFMNet, build_model
from kkbox.train import PCGrad, eval_metrics, train_model, train_pcgrad, train_uncertainty_weighted

CFG = kkbox_config.load_config()
PROCESSED_DIR = kkbox_config.abspath(CFG, CFG["paths"]["processed_dir"])
# KKBOX_MODELS_DIR/KKBOX_RESULTS_DIR let `make smoke` redirect checkpoint/results
# writes to an isolated /tmp location instead of overwriting the real committed
# artifacts - nbconvert's --output-dir only redirects where the *executed
# notebook file* is saved, it does NOT change cwd-relative paths used inside cells.
MODELS_DIR = os.environ.get("KKBOX_MODELS_DIR", kkbox_config.abspath(CFG, CFG["paths"]["models_dir"]))
RESULTS_DIR = os.environ.get("KKBOX_RESULTS_DIR", kkbox_config.abspath(CFG, CFG["paths"]["results_dir"]))
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
# Set via env var for `make smoke` (2% stratified subsample); unset for a real run.
SUBSAMPLE_FRAC = float(os.environ["KKBOX_SUBSAMPLE_FRAC"]) if "KKBOX_SUBSAMPLE_FRAC" in os.environ else None

seed_everything(CFG["seeds"]["default"])

with open(os.path.join(PROCESSED_DIR, "feature_manifest.json")) as f:
    manifest = json.load(f)

CAT_COLS, NUM_COLS, CARDINALITIES, EMBED_DIMS = columns_from_manifest(manifest)

print(f"categorical columns ({len(CAT_COLS)}): {CAT_COLS}")
print(f"numerical columns ({len(NUM_COLS)}): {NUM_COLS}")
print(f"cardinalities: {CARDINALITIES}")
print(f"embedding dims: {EMBED_DIMS}")

In [ ]:
splits = load_splits(PROCESSED_DIR)
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

if SUBSAMPLE_FRAC is not None:
    train_df = stratified_subsample(train_df, SUBSAMPLE_FRAC)
    val_df = stratified_subsample(val_df, SUBSAMPLE_FRAC)
    test_df = stratified_subsample(test_df, SUBSAMPLE_FRAC)
    print(f"SUBSAMPLE_FRAC={SUBSAMPLE_FRAC} applied (stratified by is_churn)")

train_loader = make_loader(train_df, CAT_COLS, NUM_COLS, CFG["training"]["batch_size"],
                            shuffle=True, seed=CFG["seeds"]["default"])
val_loader = make_loader(val_df, CAT_COLS, NUM_COLS, CFG["training"]["batch_size"], shuffle=False)
test_loader = make_loader(test_df, CAT_COLS, NUM_COLS, CFG["training"]["batch_size"], shuffle=False)

pos_weight = torch.tensor((train_df["is_churn"] == 0).sum() / (train_df["is_churn"] == 1).sum())
print(f"train: {len(train_df):,} rows / {len(train_loader)} batches")
print(f"val:   {len(val_df):,} rows / {len(val_loader)} batches")
print(f"test:  {len(test_df):,} rows / {len(test_loader)} batches")
print(f"pos_weight: {pos_weight.item():.2f}")

## Part 1: Model Architecture

### PyTorch `Dataset` & `DataLoader`

`CARDINALITIES` already includes the reserved "unknown" bucket from `02_Feature_Engineering.ipynb`'s label encoding (e.g. `city_enc` cardinality is 22, with index 21 reserved for missing/unseen cities), so `nn.Embedding(num_embeddings=cardinality, ...)` is used directly below — no extra `+1` is needed on top of that.

### Factorization Machine Interaction Layer

Rendle, S. (2010), *Factorization Machines*, IEEE ICDM. Computes all pairwise feature interactions in O(k·d) instead of O(d²) via the sum-of-squares-minus-square-of-sums identity.

Implementation: `kkbox.models.FMInteractionLayer` (imported above) -

```python
class FMInteractionLayer(nn.Module):
    def __init__(self, input_dim, k=8):
        super().__init__()
        self.V = nn.Parameter(torch.randn(input_dim, k) * 0.01)

    def forward(self, x):  # x shape: (batch, input_dim)
        xV = x.unsqueeze(2) * self.V.unsqueeze(0)  # (batch, input_dim, k)
        sum_then_sq = xV.sum(dim=1).pow(2)  # (batch, k)
        sq_then_sum = xV.pow(2).sum(dim=1)  # (batch, k)
        return 0.5 * (sum_then_sq - sq_then_sum)  # (batch, k)
```

### Full Model Architecture (FM + MLP + Dual Heads)

Embeddings + numerical features are concatenated (27 dims), passed through the FM layer (8-dim interaction vector), concatenated again (35 dims), then through the shared MLP backbone (Dense+BatchNorm+ReLU+Dropout × 3, ending at a 64-dim shared representation) into two independent heads.

Implementation: `kkbox.models.MultiTaskFMNet` (imported above), built via `kkbox.models.build_model(cardinalities, embed_dims, cat_cols, num_numerical, model_cfg)` which reads `fm_k`/`backbone_dims`/`dropouts`/`head_hidden_dim` from `config.yaml`'s `model` section rather than hard-coding them.

### Instantiate and inspect the architecture

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


model = build_model(CARDINALITIES, EMBED_DIMS, CAT_COLS, len(NUM_COLS), CFG["model"])
print(model)

combined_dim = sum(EMBED_DIMS.values()) + len(NUM_COLS)
print(f"\ncombined pre-FM input dim: {combined_dim} (plan: 27)")
print(f"backbone input dim (post-FM): {combined_dim + CFG['model']['fm_k']} (plan: 35)")

print("\nparameter counts:")
for name, part in [
    ("embeddings", model.embeddings),
    ("fm", model.fm),
    ("backbone", model.backbone),
    ("churn_head", model.churn_head),
    ("fwd_rev_head", model.fwd_rev_head),
]:
    print(f"  {name:12s} {count_params(part):>8,}")
print(f"  {'total':12s} {count_params(model):>8,}")

### Forward + backward pass sanity check

Pull one real batch, run it through the model, compute both losses (`BCEWithLogitsLoss` with `pos_weight` derived from the actual training-set class imbalance, and `MSELoss` on `log1p_fwd_rev_59d`), and confirm gradients flow back through the whole network. The loss values themselves are meaningless here since the model is randomly initialized — this only confirms the architecture and loss wiring are correct before any training happens.

In [ ]:
x_num, x_cat, y_churn, y_fwd_rev = next(iter(train_loader))
print(f"batch shapes: x_num={tuple(x_num.shape)}, x_cat={tuple(x_cat.shape)}, "
      f"y_churn={tuple(y_churn.shape)}, y_fwd_rev={tuple(y_fwd_rev.shape)}")

model.train()
churn_logit, fwd_rev_pred = model(x_num, x_cat)
print(f"forward output shapes: churn_logit={tuple(churn_logit.shape)}, fwd_rev_pred={tuple(fwd_rev_pred.shape)}")

print(f"pos_weight (from actual train churn rate): {pos_weight.item():.2f} (plan estimate: ~12-14)")

bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
mse_loss_fn = nn.MSELoss()
loss_churn = bce_loss_fn(churn_logit, y_churn)
loss_fwd_rev = mse_loss_fn(fwd_rev_pred, y_fwd_rev)
print(f"sanity-check losses (untrained model): bce={loss_churn.item():.4f}, mse={loss_fwd_rev.item():.4f}")

(loss_churn + loss_fwd_rev).backward()
all_grads_populated = all(p.grad is not None for p in model.parameters() if p.requires_grad)
print(f"backward() OK, all parameters have gradients: {all_grads_populated}")

## Part 2: Training & Single-Task Baselines

### Training loop (Section 6.1)

Adam (lr=1e-3, weight_decay=1e-4), gradient clipping at max_norm=1.0, `ReduceLROnPlateau` (patience=5, halves LR), early stopping (patience=10) on combined validation loss, best checkpoint saved via `torch.save`. `lambda_churn`/`lambda_fwd_rev` select which loss terms are active — the same `kkbox.train.train_model` function (all hyperparameters read from `config.yaml`'s `training` section) serves both single-task baselines here (Part 2) and the fixed-weight multi-task ablation (Exp-3..5) in Part 3 below.

### Exp-1: single-task churn baseline (`lambda_churn=1.0, lambda_fwd_rev=0.0`)

In [ ]:
seed_everything(CFG["seeds"]["default"])
model_exp1 = build_model(CARDINALITIES, EMBED_DIMS, CAT_COLS, len(NUM_COLS), CFG["model"])
hist_exp1 = train_model(
    model_exp1, train_loader, val_loader, lambda_churn=1.0, lambda_fwd_rev=0.0, pos_weight=pos_weight,
    train_cfg=CFG["training"], checkpoint_path=os.path.join(MODELS_DIR, "exp1_churn_only.pt"),
)
hist_exp1.to_csv(os.path.join(RESULTS_DIR, "exp1_history.csv"), index=False)

### Exp-2: single-task forward-revenue baseline (`lambda_churn=0.0, lambda_fwd_rev=1.0`)

In [ ]:
seed_everything(CFG["seeds"]["default"])
model_exp2 = build_model(CARDINALITIES, EMBED_DIMS, CAT_COLS, len(NUM_COLS), CFG["model"])
hist_exp2 = train_model(
    model_exp2, train_loader, val_loader, lambda_churn=0.0, lambda_fwd_rev=1.0, pos_weight=pos_weight,
    train_cfg=CFG["training"], checkpoint_path=os.path.join(MODELS_DIR, "exp2_fwd_rev_only.pt"),
)
hist_exp2.to_csv(os.path.join(RESULTS_DIR, "exp2_history.csv"), index=False)

### Loss / metric curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes[0, 0].plot(hist_exp1["epoch"], hist_exp1["train_loss"], label="train")
axes[0, 0].plot(hist_exp1["epoch"], hist_exp1["val_loss"], label="val")
axes[0, 0].set_title("Exp-1 (churn-only) loss")
axes[0, 0].legend()

axes[0, 1].plot(hist_exp1["epoch"], hist_exp1["val_auc"], color="tab:green")
axes[0, 1].set_title("Exp-1 val churn AUC")

axes[1, 0].plot(hist_exp2["epoch"], hist_exp2["train_loss"], label="train")
axes[1, 0].plot(hist_exp2["epoch"], hist_exp2["val_loss"], label="val")
axes[1, 0].set_title("Exp-2 (forward-revenue-only) loss (MSE on log1p_fwd_rev_59d)")
axes[1, 0].legend()

axes[1, 1].plot(hist_exp2["epoch"], hist_exp2["val_rmse_log"], color="tab:orange")
axes[1, 1].set_title("Exp-2 val forward-revenue RMSE (log scale)")
for ax in axes.flat:
    ax.set_xlabel("epoch")
fig.tight_layout()

### Final evaluation on val/test using the best checkpoints

Per Section 7.1 (churn) and 7.3 (forward revenue): AUC-ROC, AUC-PR for churn; RMSE/MAE/R² in both log and raw-TWD scale for forward revenue (`expm1` inverts the `log1p` target transform).

In [ ]:
model_exp1.load_state_dict(torch.load(os.path.join(MODELS_DIR, "exp1_churn_only.pt"), weights_only=True))
model_exp2.load_state_dict(torch.load(os.path.join(MODELS_DIR, "exp2_fwd_rev_only.pt"), weights_only=True))

results = {
    "exp1_churn_only": {"val": evaluate_full(model_exp1, val_loader), "test": evaluate_full(model_exp1, test_loader)},
    "exp2_fwd_rev_only": {"val": evaluate_full(model_exp2, val_loader), "test": evaluate_full(model_exp2, test_loader)},
}
with open(os.path.join(RESULTS_DIR, "baseline_results.json"), "w") as f:
    json.dump(results, f, indent=2)

results

Exp-1's forward-revenue metrics and Exp-2's churn metrics are meaningless (those heads never received gradient) — they're recorded only because the same `evaluate_full` function runs both, not because they're informative baselines.

### ROC curve (Exp-1) and predicted-vs-actual forward revenue (Exp-2)

In [ ]:
model_exp1.eval()
with torch.no_grad():
    all_logits, all_churn = [], []
    for x_num, x_cat, y_churn, y_fwd_rev in test_loader:
        churn_logit, _ = model_exp1(x_num, x_cat)
        all_logits.append(churn_logit)
        all_churn.append(y_churn)
    probs = torch.sigmoid(torch.cat(all_logits)).numpy()
    churn_true = torch.cat(all_churn).numpy()
fpr, tpr, _ = roc_curve(churn_true, probs)

model_exp2.eval()
with torch.no_grad():
    all_fwd_rev_pred, all_fwd_rev_true = [], []
    for x_num, x_cat, y_churn, y_fwd_rev in test_loader:
        _, fwd_rev_pred = model_exp2(x_num, x_cat)
        all_fwd_rev_pred.append(fwd_rev_pred)
        all_fwd_rev_true.append(y_fwd_rev)
    fwd_rev_pred_raw = np.expm1(torch.cat(all_fwd_rev_pred).numpy())
    fwd_rev_true_raw = np.expm1(torch.cat(all_fwd_rev_true).numpy())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, label=f"AUC={results['exp1_churn_only']['test']['churn_auc_roc']:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title("Exp-1 ROC curve (test)")
axes[0].legend()

sample = np.random.choice(len(fwd_rev_pred_raw), size=min(20000, len(fwd_rev_pred_raw)), replace=False)
axes[1].scatter(fwd_rev_true_raw[sample], fwd_rev_pred_raw[sample], s=2, alpha=0.15)
lims = [0, max(fwd_rev_true_raw.max(), fwd_rev_pred_raw.max())]
axes[1].plot(lims, lims, "r--", alpha=0.5)
axes[1].set_xlabel("actual forward revenue (TWD)")
axes[1].set_ylabel("predicted forward revenue (TWD)")
axes[1].set_title("Exp-2 predicted vs actual forward revenue (test)")
fig.tight_layout()

## Part 3: Multi-Task Loss-Weighting Ablation

### Exp-3, Exp-4, Exp-5: fixed loss weights

Combined loss `Total_Loss = lambda_churn * BCE + lambda_fwd_rev * MSE` (Section 4.4). Same training loop as Part 2 above, just with both lambdas nonzero.

In [ ]:
FIXED_CONFIGS = {
    "exp3_fixed_5050": (0.5, 0.5),
    "exp4_churn_dominant": (0.7, 0.3),
    "exp5_fwd_rev_dominant": (0.3, 0.7),
}

fixed_models, fixed_histories = {}, {}
for name, (lc, ll) in FIXED_CONFIGS.items():
    print(f"\n=== {name} (lambda_churn={lc}, lambda_fwd_rev={ll}) ===")
    seed_everything(CFG["seeds"]["default"])
    ckpt_path = os.path.join(MODELS_DIR, f"{name}.pt")
    model = build_model(CARDINALITIES, EMBED_DIMS, CAT_COLS, len(NUM_COLS), CFG["model"])
    hist = train_model(model, train_loader, val_loader, lambda_churn=lc, lambda_fwd_rev=ll,
                        pos_weight=pos_weight, train_cfg=CFG["training"], checkpoint_path=ckpt_path)
    hist.to_csv(os.path.join(RESULTS_DIR, f"{name}_history.csv"), index=False)
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    fixed_models[name] = model
    fixed_histories[name] = hist

### Exp-6: Uncertainty weighting (Kendall, Gal & Cipolla, 2018)

Learns a per-task log-variance `s = log(sigma^2)` and weights each loss by its learned homoscedastic uncertainty: `L = 0.5*exp(-s)*L_task + 0.5*s`. This is the standard *practical* form used for both tasks here — the paper's exact derivation differs slightly between a Gaussian (regression) and categorical (classification) likelihood, but treating the classification loss the same way as the regression term is the common simplification used in nearly all open implementations, and is what we do below. `log_var_churn` and `log_var_fwd_rev` are optimized jointly with the model in the same Adam optimizer.

In [ ]:
seed_everything(CFG["seeds"]["default"])
print("=== exp6_uncertainty ===")
exp6_ckpt = os.path.join(MODELS_DIR, "exp6_uncertainty.pt")
model_exp6, hist_exp6 = train_uncertainty_weighted(
    build_model(CARDINALITIES, EMBED_DIMS, CAT_COLS, len(NUM_COLS), CFG["model"]),
    train_loader, val_loader, CFG["training"], checkpoint_path=exp6_ckpt,
)
hist_exp6.to_csv(os.path.join(RESULTS_DIR, "exp6_history.csv"), index=False)
exp6_ckpt_data = torch.load(exp6_ckpt, weights_only=True)
model_exp6.load_state_dict(exp6_ckpt_data["model"])

### Exp-7: PCGrad (Yu et al., NeurIPS 2020)

Computes each task's gradient separately, projects away the conflicting component (negative cosine similarity) against every *other* task's *original* gradient (not the already-projected one), then sums the projected gradients before the optimizer step — see `kkbox.train.PCGrad`.

In [ ]:
seed_everything(CFG["seeds"]["default"])
print("=== exp7_pcgrad ===")
exp7_ckpt = os.path.join(MODELS_DIR, "exp7_pcgrad.pt")
model_exp7, hist_exp7 = train_pcgrad(
    build_model(CARDINALITIES, EMBED_DIMS, CAT_COLS, len(NUM_COLS), CFG["model"]),
    train_loader, val_loader, CFG["training"], checkpoint_path=exp7_ckpt,
)
hist_exp7.to_csv(os.path.join(RESULTS_DIR, "exp7_history.csv"), index=False)
model_exp7.load_state_dict(torch.load(exp7_ckpt, weights_only=True))

### Consolidated results (Section 7.4 Multi-Task Benefit Analysis)

In [ ]:
with open(os.path.join(RESULTS_DIR, "baseline_results.json")) as f:
    baseline_results = json.load(f)

all_models = {**fixed_models, "exp6_uncertainty": model_exp6, "exp7_pcgrad": model_exp7}
rows = [
    {"experiment": "exp1_churn_only", "lambda_churn": 1.0, "lambda_fwd_rev": 0.0,
     "test_auc": baseline_results["exp1_churn_only"]["test"]["churn_auc_roc"],
     "test_rmse_raw_twd": None},
    {"experiment": "exp2_fwd_rev_only", "lambda_churn": 0.0, "lambda_fwd_rev": 1.0,
     "test_auc": None,
     "test_rmse_raw_twd": baseline_results["exp2_fwd_rev_only"]["test"]["fwd_rev_rmse_raw_twd"]},
]
for name, model in all_models.items():
    test_auc, test_rmse_log = eval_metrics(model, test_loader)
    lc, ll = FIXED_CONFIGS.get(name, (None, None))
    rows.append({"experiment": name, "lambda_churn": lc, "lambda_fwd_rev": ll,
                  "test_auc": test_auc, "test_rmse_log": test_rmse_log})

results_table = pd.DataFrame(rows)
results_table.to_csv(os.path.join(RESULTS_DIR, "ablation_results_table.csv"), index=False)
results_table

### Pareto frontier: churn AUC vs. forward-revenue RMSE

Per Section 6.2's headline-result instruction: plot churn AUC and forward-revenue RMSE for Exp-1 to Exp-5 on the same axes (tracing the frontier), then overlay Exp-6/Exp-7 as starred points. A multi-task variant sitting at or near the frontier (good AUC *and* good RMSE simultaneously) is evidence that principled loss balancing recovers most of the single-task ceiling on both objectives at once.

In [ ]:
frontier_rows = []
for name, model in {**fixed_models}.items():
    auc, rmse_log = eval_metrics(model, test_loader)
    frontier_rows.append((name, auc, rmse_log))
exp1_auc = baseline_results["exp1_churn_only"]["test"]["churn_auc_roc"]
exp2_rmse_log = baseline_results["exp2_fwd_rev_only"]["test"]["fwd_rev_rmse_log"]

exp6_auc, exp6_rmse = eval_metrics(model_exp6, test_loader)
exp7_auc, exp7_rmse = eval_metrics(model_exp7, test_loader)

fig, ax = plt.subplots(figsize=(7, 5.5))
for name, auc, rmse in frontier_rows:
    ax.scatter(auc, rmse, s=90, label=name)
    ax.annotate(name.replace("exp", "E").split("_")[0], (auc, rmse), textcoords="offset points", xytext=(6, 4))
ax.scatter(exp1_auc, exp2_rmse_log, s=140, marker="D", color="black", label="single-task ceilings (E1 AUC, E2 RMSE)")
ax.scatter(exp6_auc, exp6_rmse, s=220, marker="*", color="tab:red", label="exp6_uncertainty")
ax.scatter(exp7_auc, exp7_rmse, s=220, marker="*", color="tab:purple", label="exp7_pcgrad")
ax.set_xlabel("test churn AUC-ROC (higher is better)")
ax.set_ylabel("test forward-revenue RMSE, log scale (lower is better)")
ax.set_title("Multi-task Pareto frontier: churn AUC vs. forward-revenue RMSE")
ax.legend(fontsize=8, loc="best")
ax.invert_yaxis()  # up-and-right = better on both axes
fig.tight_layout()

### Loss-weighting ablation curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, hist in {**fixed_histories, "exp6_uncertainty": hist_exp6, "exp7_pcgrad": hist_exp7}.items():
    axes[0].plot(hist["epoch"], hist["val_auc"], label=name)
    axes[1].plot(hist["epoch"], hist["val_rmse_log"], label=name)
axes[0].set_title("val churn AUC by epoch")
axes[1].set_title("val forward-revenue RMSE (log) by epoch")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend(fontsize=7)
fig.tight_layout()

### Summary

In [ ]:
mt_rows = [r for r in rows if r["experiment"] not in ("exp1_churn_only", "exp2_fwd_rev_only")]
best_auc_row = max(mt_rows, key=lambda r: r["test_auc"])
best_rmse_row = min(mt_rows, key=lambda r: r["test_rmse_log"])

print(f"Single-task ceilings: churn AUC={exp1_auc:.4f} (Exp-1), forward-revenue RMSE(log)={exp2_rmse_log:.4f} (Exp-2)")
print(f"Best multi-task churn AUC: {best_auc_row['experiment']} ({best_auc_row['test_auc']:.4f}, "
      f"{best_auc_row['test_auc'] - exp1_auc:+.4f} vs. Exp-1 ceiling)")
print(f"Best multi-task forward-revenue RMSE: {best_rmse_row['experiment']} ({best_rmse_row['test_rmse_log']:.4f}, "
      f"{best_rmse_row['test_rmse_log'] - exp2_rmse_log:+.4f} vs. Exp-2 ceiling)")
print(f"\nExp-6 (uncertainty): AUC={exp6_auc:.4f}, RMSE(log)={exp6_rmse:.4f}")
print(f"Exp-7 (PCGrad):      AUC={exp7_auc:.4f}, RMSE(log)={exp7_rmse:.4f}")

## Does the added complexity beat the CatBoost/Cox headline?

Compares the best neural-net numbers above against `03_CatBoost_and_Cox_Models.ipynb`'s
CatBoost churn classifier and forward-revenue regressor — same test set, same
`fwd_rev_59d` target, raw-TWD RMSE where both sides have it.

In [ ]:
with open(os.path.join(RESULTS_DIR, "catboost_results.json")) as f:
    catboost_results = json.load(f)

best_nn_auc = max(exp1_auc, best_auc_row["test_auc"], exp6_auc, exp7_auc)
nn_fwd_rev_rmse_raw_twd = baseline_results["exp2_fwd_rev_only"]["test"]["fwd_rev_rmse_raw_twd"]

comparison = pd.DataFrame({
    "CatBoost (primary model, 03)": {
        "churn_auc_roc": catboost_results["churn"]["test"]["churn_auc_roc"],
        "fwd_rev_rmse_raw_twd": catboost_results["fwd_rev"]["test"]["fwd_rev_rmse_raw_twd"],
    },
    "Neural FM-MLP (best across Exp-1..7)": {
        "churn_auc_roc": best_nn_auc,
        "fwd_rev_rmse_raw_twd": nn_fwd_rev_rmse_raw_twd,
    },
}).T
comparison["churn_auc_roc"] = comparison["churn_auc_roc"].round(4)
comparison["fwd_rev_rmse_raw_twd"] = comparison["fwd_rev_rmse_raw_twd"].round(2)
comparison

**Verdict**: the joint multi-task neural net does not meaningfully beat the
simpler, independently-trained CatBoost models here — despite the added
complexity of a shared FM+MLP backbone and a loss-weighting search across
Exp-3..7. This is the reason `03_CatBoost_and_Cox_Models.ipynb`, not this
notebook, is the project's primary model; this appendix exists to make that
comparison explicit and reproducible rather than asserted.